In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

  Using cached qiskit-1.2.4-cp38-abi3-macosx_11_0_arm64.whl.metadata (12 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached symengine-0.13.0.tar.gz (114 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached qiskit-1.2.4-cp38-abi3-macosx_11_0_arm64.whl (4.5 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
  error: subprocess-exited-with-error
  
  × Building wheel for symengine (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [30 lines of output]
      /opt/homebrew/Cellar/python@3.14/3.14.3_1/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/wheel/bdist_wheel.py:4: FutureWarning: The 'wheel' package is no longer the canonical location of the 'bdist_wheel' comma

In [2]:
from qiskit import QuantumCircuit, ClassicalRegister, transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.quantum_info import Statevector
import math

# =========================================================================
# BB84 Quantum Key Distribution — Plain (no attacker)
# =========================================================================
#
# BB84 (Bennett & Brassard, 1984) is a protocol by which Alice and Bob
# agree on a shared secret key over a public (quantum) channel without
# ever meeting.  This notebook simulates the protocol with no attacker
# present, confirming that Alice and Bob always derive identical keys.
#
# Agents in this notebook:
#   ALICE  — generates random bits, encodes them as qubits, sends to Bob
#   BOB    — receives qubits, measures in a random basis, reconciles key
#
# Randomness: every random choice is made by measuring |+> = H|0>,
# giving a genuinely random 0 or 1 with equal probability (Lecture 3b,
# slide 20).  Python's random module is NOT used anywhere.

simulator = BasicSimulator()

## Background: BB84 Protocol

### Bases and states (Lecture 3b, slides 13–15)

Two measurement bases are used:

| Basis | Name | Represents 0 | Represents 1 |
|-------|------|--------------|--------------|
| **Standard** (Z) | `s` | $|0\rangle$ | $|1\rangle$ |
| **Diagonal** (X) | `d` | $|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle+|1\rangle)$ | $|-\rangle = \frac{1}{\sqrt{2}}(|0\rangle-|1\rangle)$ |

### Protocol steps (Lecture 3b, slides 16–19)

1. **Alice** generates a random bit string and a random basis string.
2. Alice encodes each bit using the chosen basis and **sends the qubit to Bob** over a public quantum channel.
3. **Bob** randomly chooses a measurement basis for each qubit and measures.
4. Alice and Bob **announce their basis choices publicly** (not the bits).
5. They **keep only the bits where both chose the same basis** — those are guaranteed to agree.
6. The surviving bits form the **shared secret key** (≈ half the original length on average).

### Why this works

When Bob measures in the **same** basis Alice used, quantum mechanics guarantees he recovers Alice's bit exactly.  
When he uses the **other** basis, he gets a uniformly random result — useless for the key — so those positions are discarded.

In [3]:
# =========================================================================
# QUANTUM RANDOMNESS
# =========================================================================
# As required by the assignment, all random choices are made by measuring
# the quantum state |+> = H|0>.  This state has equal amplitudes for |0>
# and |1>, so each measurement outcome is genuinely random (Born rule).

def quantum_random_bit():
    """Return a single uniformly random bit (0 or 1) by measuring |+>."""
    qc = QuantumCircuit(1, 1)
    qc.h(0)          # |0> --H--> |+> = 1/sqrt(2)(|0> + |1>)
    qc.measure(0, 0)
    job = simulator.run(transpile(qc, simulator), shots=1)
    return int(list(job.result().get_counts().keys())[0])

def quantum_random_choice(n):
    """Choose a uniformly random index from 0..n-1 using quantum bits."""
    bits_needed = math.ceil(math.log2(n))
    while True:
        val = int(''.join(str(quantum_random_bit()) for _ in range(bits_needed)), 2)
        if val < n:   # rejection sampling keeps the distribution uniform
            return val

# Sanity check
sample = [quantum_random_bit() for _ in range(16)]
print("Sample quantum random bits:", sample)
print("(Approximately half 0s and half 1s — quantum randomness confirmed)")

Sample quantum random bits: [0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1]
(Approximately half 0s and half 1s — quantum randomness confirmed)


In [4]:
# =========================================================================
# QUBIT ENCODING AND MEASUREMENT  (Lecture 3b, slides 13–15)
# =========================================================================

BASES = ['s', 'd']   # standard (Z) and diagonal (X)

# ----- ALICE: encode a classical bit as a qubit -----

def alice_encode(bit, basis):
    """ALICE — Prepare a single qubit encoding `bit` in `basis`.

    Standard basis:  0 -> |0>,  1 -> |1>
    Diagonal basis:  0 -> |+>,  1 -> |->

    Returns a QuantumCircuit with 1 qubit, no measurements.
    (Alice 'sends' this qubit to Bob over the quantum channel.)
    """
    qc = QuantumCircuit(1)
    if bit == 1:
        qc.x(0)          # |0> -> |1>  (flip for bit=1)
    if basis == 'd':
        qc.h(0)          # |0> -> |+>, |1> -> |->
    return qc

# ----- BOB: measure the qubit Alice sent -----

def bob_measure(qubit_circuit, basis):
    """BOB — Measure the received qubit in `basis`.

    Standard basis: measure directly.
    Diagonal basis: apply H first, then measure (H converts |+>->|0>, |->->|1>).
    See Lecture 3b slide 15.

    Returns the measured bit (0 or 1).
    """
    qc = qubit_circuit.copy()
    qc.add_register(ClassicalRegister(1, 'c'))
    if basis == 'd':
        qc.h(0)          # rotate diagonal basis to standard before measuring
    qc.measure(0, 0)
    job = simulator.run(transpile(qc, simulator), shots=1)
    return int(list(job.result().get_counts().keys())[0])

# Verify: same basis -> Alice's bit recovered exactly
print("Encoding / measurement sanity check (same basis -> exact recovery):")
for bit in [0, 1]:
    for basis in ['s', 'd']:
        qc = alice_encode(bit, basis)
        recovered = bob_measure(qc, basis)
        print(f"  bit={bit}, basis={basis} -> Bob measured {recovered}  {'OK' if recovered == bit else 'FAIL'}")

print()
print("Cross-basis measurement (different basis -> random result, shown 4 times):")
for _ in range(4):
    qc = alice_encode(0, 's')
    r = bob_measure(qc, 'd')
    print(f"  Alice sent |0> (std), Bob measured in diagonal -> {r}")

Encoding / measurement sanity check (same basis -> exact recovery):
  bit=0, basis=s -> Bob measured 0  OK
  bit=0, basis=d -> Bob measured 0  OK
  bit=1, basis=s -> Bob measured 1  OK
  bit=1, basis=d -> Bob measured 1  OK

Cross-basis measurement (different basis -> random result, shown 4 times):
  Alice sent |0> (std), Bob measured in diagonal -> 0
  Alice sent |0> (std), Bob measured in diagonal -> 1
  Alice sent |0> (std), Bob measured in diagonal -> 0
  Alice sent |0> (std), Bob measured in diagonal -> 1


In [5]:
# =========================================================================
# FULL BB84 PROTOCOL — PLAIN (no attacker)
# =========================================================================

def run_bb84_plain(n_qubits=20):
    """
    Simulate the BB84 protocol without an attacker.

    Parameters
    ----------
    n_qubits : int
        Number of qubits Alice sends.  The expected key length is ~n_qubits/2.

    Returns
    -------
    alice_key, bob_key : lists of int
        The shared secret key bits (must be identical).
    """

    # ------------------------------------------------------------------
    # === ALICE: preparation phase ===
    # ------------------------------------------------------------------
    # Alice generates n_qubits random bits and n_qubits random basis
    # choices using quantum randomness, then encodes and 'sends' each qubit.

    alice_bits   = [quantum_random_bit()              for _ in range(n_qubits)]
    alice_bases  = [BASES[quantum_random_choice(2)]   for _ in range(n_qubits)]
    sent_qubits  = [alice_encode(b, bs) for b, bs in zip(alice_bits, alice_bases)]

    # ------------------------------------------------------------------
    # === BOB: measurement phase ===
    # ------------------------------------------------------------------
    # Bob randomly picks a measurement basis for each received qubit
    # (independent of Alice) and records the result.

    bob_bases   = [BASES[quantum_random_choice(2)]   for _ in range(n_qubits)]
    bob_results = [bob_measure(qc, bs) for qc, bs in zip(sent_qubits, bob_bases)]

    # ------------------------------------------------------------------
    # === PUBLIC DISCUSSION: basis reconciliation ===
    # ------------------------------------------------------------------
    # Alice and Bob announce their basis choices over a public classical
    # channel.  They keep only positions where both chose the same basis.
    # (The actual bit values are NOT revealed here.)

    matching_positions = [
        i for i in range(n_qubits)
        if alice_bases[i] == bob_bases[i]
    ]

    # ------------------------------------------------------------------
    # === KEY EXTRACTION ===
    # ------------------------------------------------------------------
    # At matching positions, Bob's measurement result equals Alice's bit.

    alice_key = [alice_bits[i]   for i in matching_positions]
    bob_key   = [bob_results[i]  for i in matching_positions]

    # ------------------------------------------------------------------
    # Print trace
    # ------------------------------------------------------------------
    header = f"{'i':>3}  {'A bit':>6}  {'A basis':>8}  {'B basis':>8}  {'B result':>9}  {'Match':>6}  {'Key bit':>8}"
    print(f"BB84 plain — {n_qubits} qubits")
    print(header)
    print("-" * len(header))
    for i in range(n_qubits):
        matched = alice_bases[i] == bob_bases[i]
        kb = str(alice_bits[i]) if matched else '-'
        print(
            f"{i:>3}  {alice_bits[i]:>6}  {alice_bases[i]:>8}  "
            f"{bob_bases[i]:>8}  {bob_results[i]:>9}  "
            f"{'YES' if matched else '---':>6}  {kb:>8}"
        )

    print()
    print(f"Matching positions : {matching_positions}")
    print(f"Alice key          : {''.join(map(str, alice_key))}")
    print(f"Bob   key          : {''.join(map(str, bob_key))}")
    print(f"Keys identical     : {alice_key == bob_key}")
    print(f"Key length         : {len(alice_key)} bits  (from {n_qubits} qubits sent)")

    # Guarantee: without an attacker, keys must always match
    assert alice_key == bob_key, "BUG: keys differ even without an attacker!"

    return alice_key, bob_key


alice_key, bob_key = run_bb84_plain(n_qubits=20)

BB84 plain — 20 qubits
  i   A bit   A basis   B basis   B result   Match   Key bit
------------------------------------------------------------
  0       1         d         d          1     YES         1
  1       1         s         s          1     YES         1
  2       1         d         d          1     YES         1
  3       0         s         s          0     YES         0
  4       1         s         d          1     ---         -
  5       0         d         d          0     YES         0
  6       1         d         s          1     ---         -
  7       0         d         d          0     YES         0
  8       1         d         d          1     YES         1
  9       0         d         s          0     ---         -
 10       1         s         s          1     YES         1
 11       1         s         s          1     YES         1
 12       1         d         s          1     ---         -
 13       0         d         d          0     YES         0
 

In [6]:
# =========================================================================
# LARGER DEMONSTRATION
# =========================================================================
# Run the protocol with more qubits to show reliable key agreement at scale.

print("Running BB84 plain with 50 qubits...\n")
key_a, key_b = run_bb84_plain(n_qubits=50)

print()
print("=" * 50)
print("RESULT SUMMARY")
print("=" * 50)
print(f"Shared key  : {''.join(map(str, key_a))}")
print(f"Key length  : {len(key_a)} bits")
print(f"Keys match  : {key_a == key_b}")
print()
print("Conclusion: Without an attacker, Alice and Bob always derive")
print("identical keys.  The key can now be used as a one-time pad.")

Running BB84 plain with 50 qubits...

BB84 plain — 50 qubits
  i   A bit   A basis   B basis   B result   Match   Key bit
------------------------------------------------------------
  0       1         s         d          0     ---         -
  1       0         s         d          1     ---         -
  2       0         d         d          0     YES         0
  3       1         s         d          0     ---         -
  4       1         d         d          1     YES         1
  5       0         s         d          1     ---         -
  6       1         s         s          1     YES         1
  7       0         d         d          0     YES         0
  8       1         d         d          1     YES         1
  9       1         d         d          1     YES         1
 10       0         s         d          0     ---         -
 11       1         s         s          1     YES         1
 12       1         s         s          1     YES         1
 13       0         s   

## Summary

This notebook demonstrates the **BB84 protocol without an attacker**:

- **Alice** prepares qubits by encoding random bits in a random basis (standard or diagonal), using quantum randomness from measuring $|+\rangle$.
- **Bob** measures each qubit in a randomly chosen basis.
- After public basis comparison, only positions where both chose the **same basis** are kept.
- At those positions, Bob's result **always equals** Alice's original bit — giving a perfectly shared secret key.
- The expected key length is approximately **half** the number of qubits sent (Lecture 3b, slide 19).

The `assert` statement confirms that the keys are **always identical** in the absence of an attacker.